# EfficientAD (Branch 4 – Anomaly Detection)

 کارهایی که این نوت‌بوک انجام می‌دهد:
- کلون کردن پیاده‌سازی غیررسمی EfficientAD (مخزن nelson1425/EfficientAD یا نسخه مشابه)
- دانلود و آماده‌سازی حداقل ۲ دیتاستِ استفاده‌شده در مقاله (**MVTec AD** و **MVTec LOCO AD**)
- (اختیاری) دانلود یک دیتاست عمومی شبیه ImageNet برای **penalty** (اینجا: **Imagenette**) 
- اجرای آموزش + تست و ذخیره **anomaly maps** (خروجی .tiff)
- ثبت تمام lossها و متریک‌های مهم در **TensorBoard** (برای Train و Test)
- اجرای اسکریپت‌های ارزیابی رسمیِ MVTec و تولید فایل‌های متریک (در صورت فعال بودن)
- ساخت یک **PDF تک‌صفحه‌ای** برای قیاس نتایج (خلاصه)
- گزارش مسیر/حجم وزن‌های ذخیره‌شده برای هر دیتاست/کلاس اجرا شده

>  `TRAIN_STEPS=70000` و `MODEL_SIZE='medium'` 


In [ ]:

# =========================
# 1) GPU Check + Libraries
# =========================
import os, sys, platform, shutil, json, re, math, time
from pathlib import Path

import torch
print("Python:", sys.version)
print("Platform:", platform.platform())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# نصب کتابخانه‌های لازم (بدون downgrade کردن torch)
!pip -q install --upgrade pip
!pip -q install tqdm tifffile scikit-learn pandas matplotlib reportlab tensorboard gdown tabulate scikit-image


In [ ]:

# =========================
# 2) Clone Repo
# =========================
%cd /content
if not Path("EfficientAD").exists():
    !git clone --depth 1 https://github.com/nelson1425/EfficientAD.git
%cd /content/EfficientAD

# نمایش فایل‌ها
!ls -la


In [ ]:

# =======================================================
# 3) Teacher weights (fallback download if missing)
# =======================================================
from pathlib import Path
import urllib.request

models_dir = Path("/content/EfficientAD/models")
models_dir.mkdir(parents=True, exist_ok=True)

hf_teacher_small = "https://huggingface.co/adghin/efficientAD_weights/resolve/main/teacher/teacher_small.pth"
hf_teacher_medium = "https://huggingface.co/adghin/efficientAD_weights/resolve/main/teacher/teacher_medium.pth"

def ensure_file(path: Path, url: str):
    if path.exists() and path.stat().st_size > 0:
        print(f"[OK] exists: {path} ({path.stat().st_size/1e6:.1f} MB)")
        return
    print(f"[DL] downloading -> {path.name}")
    urllib.request.urlretrieve(url, path.as_posix())
    print(f"[OK] downloaded: {path} ({path.stat().st_size/1e6:.1f} MB)")

# اگر فایل‌های وزن داخل ریپو نبودند، از HF دانلود می‌کنیم:
ensure_file(models_dir/"teacher_small.pth", hf_teacher_small)
ensure_file(models_dir/"teacher_medium.pth", hf_teacher_medium)

!ls -lh /content/EfficientAD/models


In [ ]:

# =======================================================
# 4) Download Datasets: MVTec AD + MVTec LOCO AD
#    (Links from the repo README)
# =======================================================
%cd /content

DATA_DIR = Path("/content/datasets")
DATA_DIR.mkdir(parents=True, exist_ok=True)

MVTEC_AD_URL = "https://www.mydrive.ch/shares/38536/3830184030e49fe74747669442f0f282/download/420938113-1629952094/mvtec_anomaly_detection.tar.xz"
MVTEC_LOCO_URL = "https://www.mydrive.ch/shares/48237/1b9106ccdfbb09a0c414bd49fe44a14a/download/430647091-1646842701/mvtec_loco_anomaly_detection.tar.xz"

def download_and_extract(url: str, out_dir: Path, out_name: str):
    out_dir.mkdir(parents=True, exist_ok=True)
    archive_path = out_dir/out_name
    if not archive_path.exists():
        print(f"[DL] {out_name}")
        !wget -q --show-progress -O "{archive_path}" "{url}"
    else:
        print(f"[OK] archive exists: {archive_path}")
    # Extract (if not already)
    marker = out_dir/(out_name + ".extracted")
    if not marker.exists():
        print("[EXTRACT] ...")
        !tar -xf "{archive_path}" -C "{out_dir}"
        marker.write_text("ok")
    else:
        print("[OK] already extracted")

download_and_extract(MVTEC_AD_URL, DATA_DIR, "mvtec_anomaly_detection.tar.xz")
download_and_extract(MVTEC_LOCO_URL, DATA_DIR, "mvtec_loco_anomaly_detection.tar.xz")

# Resolve actual roots (sometimes tar contains an extra nested folder)
def resolve_root(root: Path, expected_subfolder: str):
    # expected_subfolder مثل bottle یا breakfast_box
    if (root/expected_subfolder).exists():
        return root
    # اگر داخل root یک پوشه‌ی هم‌نام وجود داشته باشد
    for cand in root.iterdir():
        if cand.is_dir() and (cand/expected_subfolder).exists():
            return cand
    return root  # fallback

MVTEC_AD_ROOT = resolve_root(DATA_DIR/"mvtec_anomaly_detection", "bottle")
MVTEC_LOCO_ROOT = resolve_root(DATA_DIR/"mvtec_loco_anomaly_detection", "breakfast_box")

print("MVTEC_AD_ROOT:", MVTEC_AD_ROOT)
print("MVTEC_LOCO_ROOT:", MVTEC_LOCO_ROOT)

print("\n[Sample folders under MVTec AD root]")
!ls -1 "{MVTEC_AD_ROOT}" | head -n 20

print("\n[Sample folders under MVTec LOCO root]")
!ls -1 "{MVTEC_LOCO_ROOT}" | head -n 20


In [ ]:

# =======================================================
# 5) (Optional) Imagenette as ImageNet-like penalty dataset
#    EfficientAD expects a folder with subfolders of images.
# =======================================================
%cd /content/datasets

IMAGENETTE_URL = "https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz"
IMAGENETTE_TGZ = Path("imagenette2-160.tgz")
IMAGENETTE_DIR = Path("imagenette2-160")

if not IMAGENETTE_DIR.exists():
    if not IMAGENETTE_TGZ.exists():
        print("[DL] imagenette2-160.tgz")
        !wget -q --show-progress -O "{IMAGENETTE_TGZ}" "{IMAGENETTE_URL}"
    print("[EXTRACT] imagenette2-160.tgz")
    !tar -xf "{IMAGENETTE_TGZ}"
else:
    print("[OK] Imagenette already exists")

IMAGENETTE_TRAIN = (Path("/content/datasets")/IMAGENETTE_DIR/"train")
print("IMAGENETTE_TRAIN:", IMAGENETTE_TRAIN)
!ls -1 "{IMAGENETTE_TRAIN}" | head


In [ ]:

# =======================================================
# 6) Create a Colab-friendly EfficientAD script with TensorBoard
#    (Based on the original efficientad.py logic)
# =======================================================
%cd /content/EfficientAD

%%writefile efficientad_tb.py
#!/usr/bin/python
# -*- coding: utf-8 -*-

import numpy as np
import tifffile
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from torch.utils.tensorboard import SummaryWriter

import argparse
import itertools
import os
import random
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

from common import (
    get_autoencoder,
    get_pdn_small,
    get_pdn_medium,
    ImageFolderWithoutTarget,
    ImageFolderWithPath,
    InfiniteDataloader,
)

def get_argparse():
    parser = argparse.ArgumentParser()
    parser.add_argument('-d', '--dataset', default='mvtec_ad', choices=['mvtec_ad', 'mvtec_loco'])
    parser.add_argument('-s', '--subdataset', default='bottle',
                        help='One of 15 sub-datasets of Mvtec AD or 5 sub-datasets of Mvtec LOCO')
    parser.add_argument('-o', '--output_dir', default='output/1')
    parser.add_argument('-m', '--model_size', default='small', choices=['small', 'medium'])
    parser.add_argument('-w', '--weights', default='models/teacher_small.pth')
    parser.add_argument('-i', '--imagenet_train_path', default='none',
                        help='Set to "none" to disable ImageNet pretraining penalty.')
    parser.add_argument('-a', '--mvtec_ad_path', default='./mvtec_anomaly_detection',
                        help='Downloaded Mvtec AD dataset root')
    parser.add_argument('-b', '--mvtec_loco_path', default='./mvtec_loco_anomaly_detection',
                        help='Downloaded Mvtec LOCO dataset root')
    parser.add_argument('-t', '--train_steps', type=int, default=70000)

    # Colab-friendly extras
    parser.add_argument('--num_workers', type=int, default=2)
    parser.add_argument('--tb_log_every', type=int, default=10)
    parser.add_argument('--tb_eval_every', type=int, default=10000)
    parser.add_argument('--save_every', type=int, default=1000)
    parser.add_argument('--seed', type=int, default=42)
    return parser.parse_args()

# ---- constants
on_gpu = torch.cuda.is_available()
out_channels = 384
image_size = 256

# ---- transforms
default_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_ae = transforms.RandomChoice([
    transforms.ColorJitter(brightness=0.2),
    transforms.ColorJitter(contrast=0.2),
    transforms.ColorJitter(saturation=0.2)
])

def train_transform(image):
    return default_transform(image), default_transform(transform_ae(image))

@torch.no_grad()
def teacher_normalization(teacher, train_loader):
    mean_outputs = []
    for train_image, _ in tqdm(train_loader, desc='Computing mean of features'):
        if on_gpu:
            train_image = train_image.cuda(non_blocking=True)
        teacher_output = teacher(train_image)
        mean_output = torch.mean(teacher_output, dim=[0, 2, 3])
        mean_outputs.append(mean_output)
    channel_mean = torch.mean(torch.stack(mean_outputs), dim=0)
    channel_mean = channel_mean[None, :, None, None]

    mean_distances = []
    for train_image, _ in tqdm(train_loader, desc='Computing std of features'):
        if on_gpu:
            train_image = train_image.cuda(non_blocking=True)
        teacher_output = teacher(train_image)
        distance = (teacher_output - channel_mean) ** 2
        mean_distance = torch.mean(distance, dim=[0, 2, 3])
        mean_distances.append(mean_distance)
    channel_var = torch.mean(torch.stack(mean_distances), dim=0)
    channel_var = channel_var[None, :, None, None]
    channel_std = torch.sqrt(channel_var + 1e-12)
    return channel_mean, channel_std

@torch.no_grad()
def predict(image, teacher, student, autoencoder, teacher_mean, teacher_std,
            q_st_start=None, q_st_end=None, q_ae_start=None, q_ae_end=None):
    teacher_output = teacher(image)
    teacher_output = (teacher_output - teacher_mean) / teacher_std
    student_output = student(image)
    autoencoder_output = autoencoder(image)

    map_st = torch.mean((teacher_output - student_output[:, :out_channels]) ** 2, dim=1, keepdim=True)
    map_ae = torch.mean((autoencoder_output - student_output[:, out_channels:]) ** 2, dim=1, keepdim=True)

    if q_st_start is not None:
        map_st = 0.1 * (map_st - q_st_start) / (q_st_end - q_st_start + 1e-12)
    if q_ae_start is not None:
        map_ae = 0.1 * (map_ae - q_ae_start) / (q_ae_end - q_ae_start + 1e-12)

    map_combined = 0.5 * map_st + 0.5 * map_ae
    return map_combined, map_st, map_ae

@torch.no_grad()
def map_normalization(validation_loader, teacher, student, autoencoder, teacher_mean, teacher_std,
                      desc='Map normalization'):
    maps_st = []
    maps_ae = []
    # ignore augmented ae image (the second element)
    for image, _ in tqdm(validation_loader, desc=desc):
        if on_gpu:
            image = image.cuda(non_blocking=True)
        _, map_st, map_ae = predict(
            image=image, teacher=teacher, student=student, autoencoder=autoencoder,
            teacher_mean=teacher_mean, teacher_std=teacher_std
        )
        maps_st.append(map_st)
        maps_ae.append(map_ae)

    maps_st = torch.cat(maps_st)
    maps_ae = torch.cat(maps_ae)

    q_st_start = torch.quantile(maps_st, q=0.9)
    q_st_end = torch.quantile(maps_st, q=0.995)
    q_ae_start = torch.quantile(maps_ae, q=0.9)
    q_ae_end = torch.quantile(maps_ae, q=0.995)
    return q_st_start, q_st_end, q_ae_start, q_ae_end

def test(test_set, teacher, student, autoencoder, teacher_mean, teacher_std,
         q_st_start, q_st_end, q_ae_start, q_ae_end,
         test_output_dir=None, desc='Running inference'):
    y_true = []
    y_score = []

    for image, target, path in tqdm(test_set, desc=desc):
        orig_width = image.width
        orig_height = image.height

        image_t = default_transform(image)[None]
        if on_gpu:
            image_t = image_t.cuda(non_blocking=True)

        map_combined, _, _ = predict(
            image=image_t, teacher=teacher, student=student, autoencoder=autoencoder,
            teacher_mean=teacher_mean, teacher_std=teacher_std,
            q_st_start=q_st_start, q_st_end=q_st_end, q_ae_start=q_ae_start, q_ae_end=q_ae_end
        )

        map_combined = torch.nn.functional.pad(map_combined, (4, 4, 4, 4))
        map_combined = torch.nn.functional.interpolate(map_combined, (orig_height, orig_width), mode='bilinear')
        map_combined = map_combined[0, 0].detach().cpu().numpy()

        defect_class = os.path.basename(os.path.dirname(path))

        if test_output_dir is not None:
            img_nm = os.path.split(path)[1].split('.')[0]
            out_dir = Path(test_output_dir) / defect_class
            out_dir.mkdir(parents=True, exist_ok=True)
            file = out_dir / f"{img_nm}.tiff"
            tifffile.imwrite(file.as_posix(), map_combined)

        y_true_image = 0 if defect_class == 'good' else 1
        y_score_image = float(np.max(map_combined))
        y_true.append(y_true_image)
        y_score.append(y_score_image)

    auc = roc_auc_score(y_true=y_true, y_score=y_score)
    return auc * 100.0

def main():
    config = get_argparse()
    seed = config.seed
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    if config.dataset == 'mvtec_ad':
        dataset_path = config.mvtec_ad_path
    elif config.dataset == 'mvtec_loco':
        dataset_path = config.mvtec_loco_path
    else:
        raise Exception('Unknown config.dataset')

    pretrain_penalty = (config.imagenet_train_path != 'none')

    # output dirs
    train_output_dir = os.path.join(config.output_dir, 'trainings', config.dataset, config.subdataset)
    test_output_dir = os.path.join(config.output_dir, 'anomaly_maps', config.dataset, config.subdataset, 'test')
    os.makedirs(train_output_dir, exist_ok=True)
    os.makedirs(test_output_dir, exist_ok=True)

    # TensorBoard
    tb_dir = os.path.join(train_output_dir, "tensorboard")
    writer = SummaryWriter(log_dir=tb_dir)
    writer.add_text("run/config", str(vars(config)))

    # load data
    full_train_set = ImageFolderWithoutTarget(
        os.path.join(dataset_path, config.subdataset, 'train'),
        transform=transforms.Lambda(train_transform)
    )
    test_set = ImageFolderWithPath(os.path.join(dataset_path, config.subdataset, 'test'))

    if config.dataset == 'mvtec_ad':
        # paper recommends 10% validation set for MVTec AD
        train_size = int(0.9 * len(full_train_set))
        validation_size = len(full_train_set) - train_size
        rng = torch.Generator().manual_seed(seed)
        train_set, validation_set = torch.utils.data.random_split(full_train_set, [train_size, validation_size], rng)
    elif config.dataset == 'mvtec_loco':
        train_set = full_train_set
        validation_set = ImageFolderWithoutTarget(
            os.path.join(dataset_path, config.subdataset, 'validation'),
            transform=transforms.Lambda(train_transform)
        )
    else:
        raise Exception('Unknown config.dataset')

    train_loader = DataLoader(train_set, batch_size=1, shuffle=True,
                              num_workers=config.num_workers, pin_memory=True)
    train_loader_infinite = InfiniteDataloader(train_loader)
    validation_loader = DataLoader(validation_set, batch_size=1, num_workers=0)

    if pretrain_penalty:
        penalty_transform = transforms.Compose([
            transforms.Resize((2 * image_size, 2 * image_size)),
            transforms.RandomGrayscale(0.3),
            transforms.CenterCrop(image_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        penalty_set = ImageFolderWithoutTarget(config.imagenet_train_path, transform=penalty_transform)
        penalty_loader = DataLoader(penalty_set, batch_size=1, shuffle=True,
                                    num_workers=config.num_workers, pin_memory=True)
        penalty_loader_infinite = InfiniteDataloader(penalty_loader)
    else:
        penalty_loader_infinite = itertools.repeat(None)

    # create models
    if config.model_size == 'small':
        teacher = get_pdn_small(out_channels)
        student = get_pdn_small(2 * out_channels)
    elif config.model_size == 'medium':
        teacher = get_pdn_medium(out_channels)
        student = get_pdn_medium(2 * out_channels)
    else:
        raise Exception('Unknown model_size')

    # load teacher
    state_dict = torch.load(config.weights, map_location='cpu')
    teacher.load_state_dict(state_dict)
    autoencoder = get_autoencoder(out_channels)

    teacher.eval()       # frozen
    student.train()
    autoencoder.train()

    if on_gpu:
        teacher.cuda()
        student.cuda()
        autoencoder.cuda()

    teacher_mean, teacher_std = teacher_normalization(teacher, train_loader)

    optimizer = torch.optim.Adam(
        itertools.chain(student.parameters(), autoencoder.parameters()),
        lr=1e-4,
        weight_decay=1e-5
    )
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=int(0.95 * config.train_steps), gamma=0.1)

    tqdm_obj = tqdm(range(config.train_steps))
    for iteration, (image_st, image_ae), image_penalty in zip(tqdm_obj, train_loader_infinite, penalty_loader_infinite):
        if on_gpu:
            image_st = image_st.cuda(non_blocking=True)
            image_ae = image_ae.cuda(non_blocking=True)
            if image_penalty is not None:
                image_penalty = image_penalty.cuda(non_blocking=True)

        # ---- teacher/student for st loss
        with torch.no_grad():
            teacher_output_st = teacher(image_st)
            teacher_output_st = (teacher_output_st - teacher_mean) / teacher_std

        student_output_st = student(image_st)[:, :out_channels]
        distance_st = (teacher_output_st - student_output_st) ** 2
        d_hard = torch.quantile(distance_st, q=0.999)
        loss_hard = torch.mean(distance_st[distance_st >= d_hard])

        if image_penalty is not None:
            student_output_penalty = student(image_penalty)[:, :out_channels]
            loss_penalty = torch.mean(student_output_penalty ** 2)
            loss_st = loss_hard + loss_penalty
        else:
            loss_penalty = torch.tensor(0.0, device=loss_hard.device)
            loss_st = loss_hard

        # ---- AE + stae losses
        ae_output = autoencoder(image_ae)
        with torch.no_grad():
            teacher_output_ae = teacher(image_ae)
            teacher_output_ae = (teacher_output_ae - teacher_mean) / teacher_std

        student_output_ae = student(image_ae)[:, out_channels:]
        distance_ae = (teacher_output_ae - ae_output) ** 2
        distance_stae = (ae_output - student_output_ae) ** 2
        loss_ae = torch.mean(distance_ae)
        loss_stae = torch.mean(distance_stae)

        loss_total = loss_st + loss_ae + loss_stae

        optimizer.zero_grad(set_to_none=True)
        loss_total.backward()
        optimizer.step()
        scheduler.step()

        # ---- TensorBoard logging
        if iteration % config.tb_log_every == 0:
            writer.add_scalar("train/loss_total", loss_total.item(), iteration)
            writer.add_scalar("train/loss_st", loss_st.item(), iteration)
            writer.add_scalar("train/loss_hard", loss_hard.item(), iteration)
            writer.add_scalar("train/loss_penalty", loss_penalty.item(), iteration)
            writer.add_scalar("train/loss_ae", loss_ae.item(), iteration)
            writer.add_scalar("train/loss_stae", loss_stae.item(), iteration)
            writer.add_scalar("train/lr", optimizer.param_groups[0]["lr"], iteration)

        if iteration % 10 == 0:
            tqdm_obj.set_description(f"loss={loss_total.item():.4f} st={loss_st.item():.4f} ae={loss_ae.item():.4f} stae={loss_stae.item():.4f}")

        # ---- save tmp weights
        if iteration % config.save_every == 0 and iteration > 0:
            torch.save(teacher, os.path.join(train_output_dir, 'teacher_tmp.pth'))
            torch.save(student, os.path.join(train_output_dir, 'student_tmp.pth'))
            torch.save(autoencoder, os.path.join(train_output_dir, 'autoencoder_tmp.pth'))

        # ---- intermediate eval
        if iteration % config.tb_eval_every == 0 and iteration > 0:
            teacher.eval(); student.eval(); autoencoder.eval()
            q_st_start, q_st_end, q_ae_start, q_ae_end = map_normalization(
                validation_loader=validation_loader,
                teacher=teacher, student=student, autoencoder=autoencoder,
                teacher_mean=teacher_mean, teacher_std=teacher_std,
                desc='Intermediate map normalization'
            )
            auc = test(
                test_set=test_set,
                teacher=teacher, student=student, autoencoder=autoencoder,
                teacher_mean=teacher_mean, teacher_std=teacher_std,
                q_st_start=q_st_start, q_st_end=q_st_end, q_ae_start=q_ae_start, q_ae_end=q_ae_end,
                test_output_dir=None,
                desc='Intermediate inference'
            )
            print(f"Intermediate image auc: {auc:.4f}")
            writer.add_scalar("test/intermediate_image_auc", auc, iteration)
            teacher.eval(); student.train(); autoencoder.train()

    # ---- final inference + saving
    teacher.eval(); student.eval(); autoencoder.eval()

    torch.save(teacher, os.path.join(train_output_dir, 'teacher_final.pth'))
    torch.save(student, os.path.join(train_output_dir, 'student_final.pth'))
    torch.save(autoencoder, os.path.join(train_output_dir, 'autoencoder_final.pth'))

    q_st_start, q_st_end, q_ae_start, q_ae_end = map_normalization(
        validation_loader=validation_loader,
        teacher=teacher, student=student, autoencoder=autoencoder,
        teacher_mean=teacher_mean, teacher_std=teacher_std,
        desc='Final map normalization'
    )
    auc = test(
        test_set=test_set,
        teacher=teacher, student=student, autoencoder=autoencoder,
        teacher_mean=teacher_mean, teacher_std=teacher_std,
        q_st_start=q_st_start, q_st_end=q_st_end, q_ae_start=q_ae_start, q_ae_end=q_ae_end,
        test_output_dir=test_output_dir,
        desc='Final inference'
    )
    print(f"Final image auc: {auc:.4f}")
    writer.add_scalar("test/final_image_auc", auc, config.train_steps)

    # store metrics json
    metrics = {
        "dataset": config.dataset,
        "subdataset": config.subdataset,
        "model_size": config.model_size,
        "train_steps": config.train_steps,
        "final_image_auc": float(auc),
        "tb_dir": tb_dir,
        "train_output_dir": train_output_dir,
        "test_output_dir": test_output_dir,
        "pretrain_penalty": bool(pretrain_penalty),
    }
    with open(os.path.join(train_output_dir, "final_metrics.json"), "w") as f:
        import json
        json.dump(metrics, f, indent=2)

    writer.flush()
    writer.close()

if __name__ == '__main__':
    main()


In [ ]:

# =======================================================
# 7) Run EfficientAD on >=2 datasets (MVTec AD + LOCO)
#    انتخاب دسته‌ها:
#    - برای اجرای کامل، لیست FULL را استفاده کن.
#    - برای تست سریع پایپ‌لاین، لیست QUICK را.
# =======================================================
%cd /content/EfficientAD

from pathlib import Path
import subprocess, json, pandas as pd

# ---- تنظیمات اصلی
OUTPUT_DIR = Path("/content/outputs/efficientad_runs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_SIZE = "medium"      # "small" یا "medium"
TRAIN_STEPS = 70000        # برای تست سریع: مثلا 2000 یا 10000
USE_PENALTY = True         # اگر False شود، --imagenet_train_path none می‌شود.

TEACHER_WEIGHTS = "/content/EfficientAD/models/teacher_medium.pth" if MODEL_SIZE=="medium" else "/content/EfficientAD/models/teacher_small.pth"
IMAGENET_TRAIN_PATH = str(IMAGENETTE_TRAIN) if USE_PENALTY else "none"

# ---- دسته‌های پیشنهادی (برای تست سریع)
MVTEC_AD_OBJECTS_QUICK = ["bottle", "cable"]
MVTEC_LOCO_OBJECTS_QUICK = ["breakfast_box", "juice_bottle"]

# ---- اجرای کامل (تمام کلاس‌ها)
MVTEC_AD_OBJECTS_FULL = ["carpet","grid","leather","tile","wood","bottle","cable","capsule","hazelnut","metal_nut","pill","screw","toothbrush","transistor","zipper"]
MVTEC_LOCO_OBJECTS_FULL = ["breakfast_box","juice_bottle","pushpins","screw_bag","splicing_connectors"]

# انتخاب لیست نهایی (اینجا روی QUICK گذاشتم؛ اگر می‌خواهی کامل اجرا شود FULL را جایگزین کن)
MVTEC_AD_OBJECTS = MVTEC_AD_OBJECTS_QUICK
MVTEC_LOCO_OBJECTS = MVTEC_LOCO_OBJECTS_QUICK

def run_one(dataset: str, subdataset: str, dataset_root: str):
    cmd = [
        "python", "efficientad_tb.py",
        "--dataset", dataset,
        "--subdataset", subdataset,
        "--output_dir", str(OUTPUT_DIR),
        "--model_size", MODEL_SIZE,
        "--weights", TEACHER_WEIGHTS,
        "--train_steps", str(TRAIN_STEPS),
        "--num_workers", "2",
        "--tb_log_every", "10",
        "--tb_eval_every", "10000",
        "--save_every", "1000",
    ]
    if dataset == "mvtec_ad":
        cmd += ["--mvtec_ad_path", dataset_root]
    else:
        cmd += ["--mvtec_loco_path", dataset_root]

    if USE_PENALTY:
        cmd += ["--imagenet_train_path", IMAGENET_TRAIN_PATH]
    else:
        cmd += ["--imagenet_train_path", "none"]

    print("\nRUN:", " ".join(cmd))
    subprocess.run(cmd, check=True)

# ---- اجرا روی دیتاست 1: MVTec AD
for obj in MVTEC_AD_OBJECTS:
    run_one("mvtec_ad", obj, str(MVTEC_AD_ROOT))

# ---- اجرا روی دیتاست 2: MVTec LOCO AD
for obj in MVTEC_LOCO_OBJECTS:
    run_one("mvtec_loco", obj, str(MVTEC_LOCO_ROOT))

print("\n[DONE] training + inference finished. Outputs:", OUTPUT_DIR)


In [ ]:

# =======================================================
# 8) TensorBoard (Train & Test)
# =======================================================
%load_ext tensorboard
TB_ROOT = "/content/outputs/efficientad_runs/trainings"
%tensorboard --logdir {TB_ROOT}


In [ ]:

# =======================================================
# 9) Collect metrics + weights report (CSV)
# =======================================================
import pandas as pd
from pathlib import Path
import json

TRAININGS_ROOT = OUTPUT_DIR/"trainings"
rows = []
weights_rows = []

for dataset in ["mvtec_ad", "mvtec_loco"]:
    ds_dir = TRAININGS_ROOT/dataset
    if not ds_dir.exists():
        continue
    for sub in sorted([p.name for p in ds_dir.iterdir() if p.is_dir()]):
        metrics_path = ds_dir/sub/"final_metrics.json"
        if metrics_path.exists():
            m = json.loads(metrics_path.read_text())
        else:
            m = {"dataset": dataset, "subdataset": sub}
        rows.append(m)

        # weights
        w_teacher = ds_dir/sub/"teacher_final.pth"
        w_student = ds_dir/sub/"student_final.pth"
        w_ae = ds_dir/sub/"autoencoder_final.pth"
        for wpath in [w_teacher, w_student, w_ae]:
            if wpath.exists():
                weights_rows.append({
                    "dataset": dataset,
                    "subdataset": sub,
                    "weight_file": wpath.name,
                    "path": str(wpath),
                    "size_mb": round(wpath.stat().st_size/1e6, 2),
                })

metrics_df = pd.DataFrame(rows).sort_values(["dataset","subdataset"])
weights_df = pd.DataFrame(weights_rows).sort_values(["dataset","subdataset","weight_file"])

metrics_csv = OUTPUT_DIR/"metrics_summary.csv"
weights_csv = OUTPUT_DIR/"weights_report.csv"

metrics_df.to_csv(metrics_csv, index=False)
weights_df.to_csv(weights_csv, index=False)

print("Saved:", metrics_csv, weights_csv)
display(metrics_df)
display(weights_df.head(20))


In [ ]:

# =======================================================
# 9.5) Custom pixel metrics (Pixel AUROC + AU-PRO@0.3)
#      بدون نیاز به اسکریپت‌های ارزیابی خارجی
#      - برای Train/Test در TensorBoard هم ثبت می‌کند.
# =======================================================
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import tifffile
from sklearn.metrics import roc_auc_score
from skimage.measure import label
from torch.utils.tensorboard import SummaryWriter
import json, math

def _load_mask(mask_path: Path, target_shape):
    if mask_path.exists():
        m = np.array(Image.open(mask_path).convert("L"), dtype=np.uint8)
        m = (m > 0).astype(np.uint8)
        if m.shape != target_shape:
            m_img = Image.fromarray(m * 255).resize((target_shape[1], target_shape[0]), resample=Image.NEAREST)
            m = (np.array(m_img) > 0).astype(np.uint8)
        return m
    return np.zeros(target_shape, dtype=np.uint8)

def pixel_auroc_and_pro(pred_maps, gt_masks, pro_limit=0.3, n_thresh=200):
    # Pixel AUROC
    y_true = np.concatenate([m.flatten() for m in gt_masks]).astype(np.uint8)
    y_score = np.concatenate([p.flatten() for p in pred_maps]).astype(np.float32)

    if len(np.unique(y_true)) < 2:
        pixel_auc = np.nan
    else:
        pixel_auc = float(roc_auc_score(y_true, y_score))

    # AU-PRO@limit
    s_min, s_max = float(y_score.min()), float(y_score.max())
    if s_max - s_min < 1e-12:
        return pixel_auc, np.nan
    pred_maps_n = [(p - s_min) / (s_max - s_min + 1e-12) for p in pred_maps]

    thresholds = np.linspace(0.0, 1.0, n_thresh, dtype=np.float32)
    fprs, pros = [], []

    for thr in thresholds:
        pro_vals = []
        fp = 0
        tn = 0

        for p, gt in zip(pred_maps_n, gt_masks):
            pred_bin = (p >= thr).astype(np.uint8)

            bg = (gt == 0)
            fp += int((pred_bin[bg] == 1).sum())
            tn += int((pred_bin[bg] == 0).sum())

            if gt.sum() == 0:
                continue

            cc = label(gt, connectivity=1)
            for rid in range(1, cc.max() + 1):
                region = (cc == rid)
                region_area = int(region.sum())
                if region_area == 0:
                    continue
                inter = int((pred_bin[region] == 1).sum())
                pro_vals.append(inter / (region_area + 1e-12))

        fpr = fp / (fp + tn + 1e-12)
        pro = float(np.mean(pro_vals)) if len(pro_vals) else 0.0

        fprs.append(fpr)
        pros.append(pro)

    fprs = np.array(fprs, dtype=np.float32)
    pros = np.array(pros, dtype=np.float32)

    order = np.argsort(fprs)
    fprs = fprs[order]
    pros = pros[order]

    m = fprs <= pro_limit
    if m.sum() < 2:
        au_pro = np.nan
    else:
        area = np.trapz(pros[m], fprs[m])
        au_pro = float(area / (pro_limit + 1e-12))

    return pixel_auc, au_pro

def evaluate_one(dataset_root: Path, anomaly_maps_root: Path, subdataset: str):
    test_dir = dataset_root / subdataset / "test"
    gt_dir = dataset_root / subdataset / "ground_truth"

    pred_maps, gt_masks = [], []

    for defect in sorted([p.name for p in test_dir.iterdir() if p.is_dir()]):
        img_dir = test_dir / defect
        for img_path in sorted(img_dir.glob("*.*")):
            img_stem = img_path.stem
            pred_path = anomaly_maps_root / defect / f"{img_stem}.tiff"
            if not pred_path.exists():
                alt = anomaly_maps_root / defect / f"{img_stem}.tif"
                if alt.exists():
                    pred_path = alt
                else:
                    continue

            pred = tifffile.imread(pred_path.as_posix()).astype(np.float32)
            if pred.ndim != 2:
                pred = pred.squeeze()

            if defect == "good":
                gt = np.zeros(pred.shape, dtype=np.uint8)
            else:
                mask_path = gt_dir / defect / f"{img_stem}_mask.png"
                gt = _load_mask(mask_path, pred.shape)

            pred_maps.append(pred)
            gt_masks.append(gt)

    if len(pred_maps) == 0:
        return {"pixel_auroc": np.nan, "au_pro_0.3": np.nan, "n_images": 0}

    pixel_auc, au_pro = pixel_auroc_and_pro(pred_maps, gt_masks, pro_limit=0.3, n_thresh=200)
    return {"pixel_auroc": pixel_auc, "au_pro_0.3": au_pro, "n_images": len(pred_maps)}

TRAININGS_ROOT = OUTPUT_DIR / "trainings"
ANOMALY_ROOT = OUTPUT_DIR / "anomaly_maps"

extra_rows = []
for dataset in ["mvtec_ad", "mvtec_loco"]:
    ds_dir = TRAININGS_ROOT / dataset
    if not ds_dir.exists():
        continue

    dataset_root = Path(MVTEC_AD_ROOT) if dataset == "mvtec_ad" else Path(MVTEC_LOCO_ROOT)

    for sub in sorted([p.name for p in ds_dir.iterdir() if p.is_dir()]):
        maps_root = ANOMALY_ROOT / dataset / sub / "test"
        res = evaluate_one(dataset_root, maps_root, sub)

        tb_dir = ds_dir / sub / "tensorboard"
        writer = SummaryWriter(log_dir=str(tb_dir))
        if not (res["pixel_auroc"] is None or math.isnan(res["pixel_auroc"])):
            writer.add_scalar("test/pixel_auroc_custom", res["pixel_auroc"], TRAIN_STEPS)
        if not (res["au_pro_0.3"] is None or math.isnan(res["au_pro_0.3"])):
            writer.add_scalar("test/au_pro_0.3_custom", res["au_pro_0.3"], TRAIN_STEPS)
        writer.flush()
        writer.close()

        extra_rows.append({"dataset": dataset, "subdataset": sub, **res})

extra_df = pd.DataFrame(extra_rows).sort_values(["dataset", "subdataset"])
extra_csv = OUTPUT_DIR / "pixel_metrics_custom.csv"
extra_df.to_csv(extra_csv, index=False)
print("Saved:", extra_csv)
display(extra_df)


In [ ]:

# =======================================================
# 10) (Optional) Run official evaluation scripts (MVTec AD / LOCO)
#     و استخراج متریک‌ها برای گزارش PDF
# =======================================================
# اگر این مرحله را نمی‌خواهی، می‌توانی از آن عبور کنی و فقط از image-AUC استفاده کنی.

%cd /content

EVAL_DIR = Path("/content/eval")
EVAL_DIR.mkdir(parents=True, exist_ok=True)

MVTEC_AD_EVAL_URL = "https://www.mydrive.ch/shares/60736/698155e0e6d0467c4ff6203b16a31dc9/download/439517473-1665667812/mvtec_ad_evaluation.tar.xz"
MVTEC_LOCO_EVAL_URL = "https://www.mydrive.ch/shares/48245/a4e9922c5efa93f57b6a0ff9f5c6b969/download/430648014-1646847095/mvtec_loco_ad_evaluation.tar.xz"

def dl_extract_eval(url, name):
    arc = EVAL_DIR/name
    if not arc.exists():
        !wget -q --show-progress -O "{arc}" "{url}"
    marker = EVAL_DIR/(name + ".extracted")
    if not marker.exists():
        !tar -xf "{arc}" -C "{EVAL_DIR}"
        marker.write_text("ok")

dl_extract_eval(MVTEC_AD_EVAL_URL, "mvtec_ad_evaluation.tar.xz")
dl_extract_eval(MVTEC_LOCO_EVAL_URL, "mvtec_loco_ad_evaluation.tar.xz")

# Patch for .tiff extension if needed (safe even if already supports it)
def patch_tiff_reader(file_path: Path):
    if not file_path.exists():
        return
    txt = file_path.read_text()
    if "exts=['.tiff'" in txt or 'exts=[".tiff"' in txt:
        return
    # try to patch read_tiff default exts list
    txt2 = re.sub(r"exts\s*=\s*\(\s*'\.tif'\s*\)", "exts=('.tif','.tiff')", txt)
    txt2 = re.sub(r"exts\s*=\s*\[\s*'\.tif'\s*\]", "exts=['.tif','.tiff']", txt2)
    if txt2 != txt:
        file_path.write_text(txt2)

# Attempt patch on common util files (best effort)
patch_tiff_reader(EVAL_DIR/"mvtec_ad_evaluation"/"evaluation"/"generic_util.py")

print("Eval dirs:")
!find "{EVAL_DIR}" -maxdepth 2 -type d -print


In [ ]:

# =======================================================
# 11) Evaluate MVTec AD categories executed (produces metrics json)
# =======================================================
import subprocess, json
from pathlib import Path

ANOMALY_MAPS_AD = OUTPUT_DIR/"anomaly_maps"/"mvtec_ad"
DATASET_AD = Path(MVTEC_AD_ROOT)

OUT_METRICS_DIR = OUTPUT_DIR/"metrics"
OUT_METRICS_DIR.mkdir(parents=True, exist_ok=True)

ad_eval_script = EVAL_DIR/"mvtec_ad_evaluation"/"evaluate_experiment.py"

# eval only categories you trained
for obj in MVTEC_AD_OBJECTS:
    cmd = [
        "python", str(ad_eval_script),
        "--dataset_base_dir", str(DATASET_AD),
        "--anomaly_maps_dir", str(ANOMALY_MAPS_AD),
        "--output_dir", str(OUT_METRICS_DIR),
        "--evaluated_objects", obj,
        "--pro_integration_limit", "0.3"
    ]
    print("\nEVAL:", " ".join(cmd))
    subprocess.run(cmd, check=True)

print("\nMVTec AD evaluation done.")
!find "{OUT_METRICS_DIR}" -maxdepth 3 -type f -name "*.json" -print


In [ ]:

# =======================================================
# 12) Evaluate MVTec LOCO categories executed (produces metrics json)
# =======================================================
import subprocess
from pathlib import Path

ANOMALY_MAPS_LOCO = OUTPUT_DIR/"anomaly_maps"/"mvtec_loco"
DATASET_LOCO = Path(MVTEC_LOCO_ROOT)

loco_eval_script = EVAL_DIR/"mvtec_loco_ad_evaluation"/"evaluate_experiment.py"

for obj in MVTEC_LOCO_OBJECTS:
    cmd = [
        "python", str(loco_eval_script),
        "--dataset_base_dir", str(DATASET_LOCO),
        "--anomaly_maps_dir", str(ANOMALY_MAPS_LOCO),
        "--output_dir", str(OUT_METRICS_DIR),
        "--object_name", obj,
    ]
    print("\nEVAL:", " ".join(cmd))
    subprocess.run(cmd, check=True)

print("\nMVTec LOCO evaluation done.")
!find "{OUT_METRICS_DIR}" -maxdepth 4 -type f -name "*.json" -print


In [ ]:

# =======================================================
# 13) Build 1-page PDF comparison report
#     - خلاصه نتایج شما + مقایسه با اعداد گزارش‌شده در مقاله/README
# =======================================================
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import json, glob, math

# Load our metrics (image auc from final_metrics.json)
metrics_df = pd.read_csv(OUTPUT_DIR/"metrics_summary.csv")
pixel_csv = OUTPUT_DIR/"pixel_metrics_custom.csv"
pixel_df = pd.read_csv(pixel_csv) if pixel_csv.exists() else None


# Paper reference (EfficientAD WACV 2024) — mean image AUROC numbers commonly reported for EfficientAD-M
# (اگر مدل small اجرا کردی، این اعداد را مطابق paper/README تغییر بده.)
paper_ref = {
    ("mvtec_ad", "medium"): 99.1,
    ("mvtec_loco", "medium"): 90.7,
}

# Compute mean over executed categories (not necessarily full dataset mean if you didn't run all)
summary_rows = []
for dataset in metrics_df["dataset"].unique():
    sub = metrics_df[metrics_df["dataset"]==dataset]
    mean_auc = sub["final_image_auc"].mean()
mean_pixel = None
mean_pro = None
if pixel_df is not None and len(pixel_df)>0:
    psub = pixel_df[pixel_df["dataset"]==dataset]
    if len(psub)>0:
        mean_pixel = float(psub["pixel_auroc"].mean())
        mean_pro = float(psub["au_pro_0.3"].mean())

    n = len(sub)
    ref = paper_ref.get((dataset, MODEL_SIZE), None)
    summary_rows.append({
        "dataset": dataset,
        "model_size": MODEL_SIZE,
        "categories_executed": n,
        "our_mean_image_auc": round(mean_auc, 3),
        "paper_mean_image_auc": ref,
        "gap": (round(mean_auc-ref, 3) if ref is not None else None),
        "our_mean_pixel_auroc": (round(mean_pixel, 3) if mean_pixel is not None and not math.isnan(mean_pixel) else None),
        "our_mean_au_pro@0.3": (round(mean_pro, 3) if mean_pro is not None and not math.isnan(mean_pro) else None)
    })

summary_df = pd.DataFrame(summary_rows)

# Make 1-page PDF
pdf_path = OUTPUT_DIR/"comparison_one_page.pdf"

fig = plt.figure(figsize=(11.69, 8.27))  # A4 landscape
plt.axis("off")
title = "EfficientAD – Verification Report (1 page)\n"
subtitle = f"Model: {MODEL_SIZE} | Train steps: {TRAIN_STEPS} | Penalty: {USE_PENALTY}"
plt.text(0.01, 0.95, title, fontsize=16, weight="bold")
plt.text(0.01, 0.90, subtitle, fontsize=11)

tbl = plt.table(
    cellText=summary_df.values,
    colLabels=summary_df.columns,
    loc="center",
    cellLoc="center"
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 1.6)

note = ("Note:\n"
        "- our_mean_image_auc is averaged over the categories YOU executed.\n"
        "- paper_mean_image_auc refers to mean over all categories in the official benchmark.\n"
        "- To reproduce paper numbers, run FULL category lists with TRAIN_STEPS=70000.")
plt.text(0.01, 0.08, note, fontsize=9)

plt.savefig(pdf_path, bbox_inches="tight")
plt.close(fig)

print("Saved PDF:", pdf_path)
print("You can download it from the file browser in Colab.")
